# P8-QUICK — SpeedNet 성능 가늠용 (목표 15~25분)

`code_p8.ipynb` 의 **측정 전용 축소판**이다. 제출물을 만들지 않는다.
목적은 하나다 — **논문의 SpeedNet(CNN-LSTM)이 맵 없는 격자 피처 대조군을 이기는가?**

## 무엇을 잘랐나

| 항목 | 전체판 | 여기 | 이유 |
|---|---|---|---|
| PNG 디코딩 | train+val+test 15,662 프레임 | **train 서브셋만** (약 3~4천) | 가장 큰 고정비용. 25분 → 5분 |
| 투영 맵 | 60×120 | **40×80** | Conv 계산량 2.2배 감소 |
| Conv 필터 | 32/64/128/128 | **16/32/64/64** | 계산량 2.5배 감소 |
| 폴드 × epoch | 2 × 20 (+5×30 재확인) | **2 × 10** | |
| 최종 학습·추론·제출 | 있음 | **없음** | 측정만 한다 |
| Grad-CAM | 있음 | 없음 | 제출과 무관 |

**자르지 않은 것:** 논문의 전처리 파이프라인(Stonyhurst 투영 / Eq. 1 고정상수 표준화 /
동적 임계 이진맵), SpeedNet 의 위상(Conv×4 → LSTM → FFNN), 사슬 단위 CV, 대회 지표 정렬.
즉 **구조는 그대로이고 크기만 줄였다.**

## 🔴 결과를 읽는 법 — 이게 제일 중요하다

**절대 RMSE 숫자를 전체판·P3(58.80)과 비교하지 말 것.** 학습 데이터가 1/3 이라 무조건 나쁘게 나온다.
읽어야 할 것은 **같은 표 안에서의 상대 순서와 격차**다.

1. **`A1. feature` 대비 `B/C/D` 의 차이** — 이 노트북의 유일한 결론이다
2. **persistence 대비 개선폭** — 데이터 크기가 달라도 비교적 잘 전이되는 양이다
3. **`fold_se`** — 폴드 2개뿐이라 표준오차가 크다. `fold_se × 2` 보다 작은 차이는 **아무 의미 없다**

### 이 축소판은 CNN 에게 불리하다 (반드시 감안할 것)

CNN 은 격자 피처보다 데이터에 훨씬 굶주린 모델이다. 표본을 1/3 로 줄이고 필터도 절반으로
줄였으므로, 여기서 나온 SpeedNet 점수는 **전체 데이터에서 낼 수 있는 성능의 하한**이다.

| 여기서 나온 결과 | 해석 | 다음 행동 |
|---|---|---|
| CNN 이 `feature` 를 **이긴다** | 강한 신호. 불리한 조건에서도 이겼다 | 전체판을 돌릴 가치가 충분하다 |
| **비긴다** (차이 < `fold_se`×2) | 판단 불가 | 시간이 있으면 전체판, 없으면 P3 유지 |
| CNN 이 **크게 진다** (> 5 km/s) | 데이터 부족만으로 설명하기 어렵다 | 전체판에 시간을 쓰지 말 것. HANDOFF §4 의 P6 실패와 같은 방향 |

### 이 노트북이 답하지 않는 것

- test/Public 성능 (추론을 하지 않는다)
- 최적 epoch 수 (10 epoch 로 잘랐으므로 수렴 전일 수 있다 — 학습곡선이 끝까지 내려가고
  있으면 `LADDER_EPOCHS` 를 늘려야 한다는 뜻이다)
- **투영 자체의 이득** — 진짜 일면 면적, 중앙자오선 밴드 정의는 A0 에도 이미 들어 있어
  이 사다리로 분리되지 않는다. A0 vs A1 은 경도 외삽의 **일부**만 잰다 (§3 주의 참조)

## 실행 순서

셀을 위에서 아래로 한 번씩. `Shift+Enter` 로 하나씩 (Run 연타 금지 — 진행상황을 못 본다).
마지막 셀이 요약표와 **"전체판을 돌릴 가치가 있는가"** 판정을 출력한다.

In [ ]:
from pathlib import Path
import gc, math, os, random, time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 777
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT_CANDIDATES = [
    Path(os.getenv("SW_DATA_ROOT", "")) if os.getenv("SW_DATA_ROOT") else None,
    Path("public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public_dataset/competition_dataset_6h"),
    Path("public/public_dataset/competition_dataset_6h"),
    Path("/home/jovyan/public/public_dataset/competition_dataset_6h"),
    Path("dataset"), Path("/home/jovyan/dataset"),
]
DATA_ROOT = None
for candidate in DATA_ROOT_CANDIDATES:
    if candidate is not None and (candidate / "train/inputs.csv").exists():
        DATA_ROOT = candidate
        break
if DATA_ROOT is None:
    raise FileNotFoundError("데이터 경로 없음")

CACHE_ROOT = Path("work/cache"); CACHE_ROOT.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = Path("work/outputs_p8quick"); OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_COLUMNS = [f"image_{i:02d}" for i in range(20)]
WIND_COLUMNS = [f"wind_{i:02d}" for i in range(20)]
TARGET_COLUMNS = [f"target_{i:02d}" for i in range(12)]
HORIZONS = np.arange(1, 13) * 6
CHANNELS = ("193", "211")
AU_KM = 1.496e8
STEP_HOURS = 6.0

# ── ⏱ 시간을 지배하는 3개 손잡이 ───────────────────────────────────────────
SUBSET_SAMPLES = 4000      # train 9,607 중 쓸 샘플 수. 추출 시간이 여기에 정비례한다
LADDER_FOLDS = 2           # 사다리 폴드 수
LADDER_EPOCHS = 10         # 사다리 epoch 수
# 더 빨리 보고 싶으면 SUBSET_SAMPLES=2500, LADDER_EPOCHS=8 로. 대신 fold_se 가 커진다.

# ── 논문 §2.1 Stonyhurst 투영 (해상도만 축소) ───────────────────────────────
PROJ_CODE_VERSION = "p8q"
MAP_LAT, MAP_LON = 36, 72
LAT_RANGE_DEG, LON_RANGE_DEG = 60.0, 60.0
B0_DEG = 0.0               # 천체력 = 외부 데이터라 0 고정
DISK_MARGIN = 0.995
SIDEREAL_DEG_PER_HOUR = 14.18 / 24.0

# ── 논문 §2.1 동적 임계 / 격자 ─────────────────────────────────────────────
CH_CUTS = (0.30, 0.45, 0.60)
BRIGHT_CUT = 1.60
BM_CHANNEL = "and"
FINE_GRID = (12, 24)       # MAP 을 나눠야 한다 (36/12=3, 72/24=3)
CH_GRID = (6, 3)           # 전체판과 동일하게 유지 (12/6=2, 24/3=8)
FOLD_LATITUDE = True
USE_LEVELS = ("ch0.45", "bright")

# ── 탄도 정렬 + 논문 §4.3 중앙자오선 ───────────────────────────────────────
REFERENCE_TRANSIT_HOURS = 108.0
TRANSIT_SPEEDS = (315.0, 385.0, 500.0, 600.0)
BALLISTIC_OFFSETS = (-4, -2, 0, 2)
BALLISTIC_SOURCE = "window"
MERIDIAN_HALF_WIDTH_DEG = 10.0
USE_ROTATION_EXTRAPOLATION = True

# ── 모델 (필터·유닛 축소) ──────────────────────────────────────────────────
MODEL_VARIANT = "feature"
MAP_CHANNELS = ("193", "211", "bm")
SPEEDNET_FILTERS = (16, 32, 64, 64)
SPEEDNET_KERNEL = 3
SPEEDNET_POOL = None       # conv 출력 그대로. 36x72 -> 18x36 -> 9x18 -> 4x9 -> 2x4
LSTM_UNITS = 100           # 논문 Table 4
FFNN_UNITS = 200           # 논문 Table 4
CH_HIDDEN = 64
CH_BIDIRECTIONAL = True
GATHER_DIM = 16
HORIZON_EMBED = 8
DROPOUT = 0.4
RESIDUAL_OUTPUT = True
VERBOSE_MODEL = False

# ── 학습 ───────────────────────────────────────────────────────────────────
BATCH_SIZE = 64
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-3
LOSS_KIND = "metric"       # 대회 지표 정렬. 논문 원형은 "mse"
GRAD_CLIP = 1.0
NUM_WORKERS = 4
LOSS_EPSILON = 1e-8
LOSS_SCALE = 100.0
AUGMENT = True
AUG_CH_NOISE = 0.05
AUG_MAP_GAIN = 0.05
AUG_MAP_SHIFT = 2

HSE_THRESHOLD, HSE_LOOKBACK = 50.0, 4

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
PIN_MEMORY = DEVICE.type == "cuda"
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True

STAGE_TIME = {}


def stage(name):
    """단계별 소요 시간을 기록한다. 전체판 예산을 잡는 근거가 된다."""
    STAGE_TIME[name] = time.perf_counter()
    return STAGE_TIME[name]


print("PyTorch:", torch.__version__, "| device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print(f"투영 맵 {MAP_LAT}x{MAP_LON} | 서브셋 목표 {SUBSET_SAMPLES:,} 샘플 | "
      f"사다리 {LADDER_FOLDS} fold x {LADDER_EPOCHS} epoch")

## 1. 데이터 · 시간축 복원 · 서브셋 선택

`inputs.csv` 에 타임스탬프가 없고 행 순서도 시간 순서가 아니다. 한 행의 `image_00..image_19` 가
연속 20시점이라는 사실만으로 **프레임 사슬**(연속 관측 구간)을 복원한다. 사슬이 CV 폴드의 단위다.

### 서브셋은 사슬을 **일정 간격으로** 고른다

앞쪽 몇 년만 잘라 쓰면 태양주기 위상이 한쪽으로 쏠린다. 사슬을 `stride` 간격으로 건너뛰며 골라
전 기간에 고르게 퍼뜨린다. 사슬을 **통째로** 배정하므로 20스텝 윈도우 중첩에 의한 누수는 없다.

> validation·test 는 **읽지도 않는다.** 이 노트북은 추론을 하지 않으므로 그 PNG 5,523장을
> 디코딩할 이유가 없다. 전체 고정비용의 35% 가 여기서 사라진다.

In [ ]:
stage("start")
train_inputs = pd.read_csv(DATA_ROOT / "train/inputs.csv")
train_targets_frame = pd.read_csv(DATA_ROOT / "train/targets.csv")
assert train_inputs.sample_id.tolist() == train_targets_frame.sample_id.tolist()

wind_raw = train_inputs[WIND_COLUMNS].to_numpy(np.float64)
WIND_VALID_ALL = np.isfinite(wind_raw).astype(np.float32)
WIND_ALL = np.nan_to_num(pd.DataFrame(wind_raw).ffill(axis=1).bfill(axis=1).to_numpy(np.float32),
                         nan=float(np.nanmedian(wind_raw)))
TARGETS_ALL = train_targets_frame[TARGET_COLUMNS].to_numpy(np.float32)


def reconstruct_frame_chains(inputs):
    """이미지 파일명만으로 프레임 시간축을 복원한다. 행 순서에 의존하지 않는다."""
    images = inputs[IMAGE_COLUMNS].to_numpy()
    successor, predecessor, conflicts = {}, {}, 0
    for row in images:
        for current, following in zip(row[:-1], row[1:]):
            if successor.setdefault(current, following) != following:
                conflicts += 1
            if predecessor.setdefault(following, current) != current:
                conflicts += 1
    names = set(images.ravel().tolist())
    chains, visited = [], set()
    for head in sorted(names - set(predecessor)):
        chain, node = [], head
        while node is not None and node not in visited:
            visited.add(node); chain.append(node); node = successor.get(node)
        chains.append(chain)
    assert conflicts == 0 and not (names - visited), "사슬 복원 실패"
    return chains


CHAINS = reconstruct_frame_chains(train_inputs)
_position = {name: c for c, chain in enumerate(CHAINS) for name in chain}
CHAIN_OF_ROW = np.array([_position[n] for n in train_inputs[IMAGE_COLUMNS[0]].to_numpy()])
COUNTS = np.bincount(CHAIN_OF_ROW, minlength=len(CHAINS))

# ── 서브셋: 전 기간에 고르게 퍼지도록 사슬을 stride 간격으로 고른다 ─────────
stride = 1
for candidate_stride in range(1, len(CHAINS) + 1):
    picked = np.arange(0, len(CHAINS), candidate_stride)
    if COUNTS[picked].sum() <= SUBSET_SAMPLES:
        stride = candidate_stride
        break
PICKED_CHAINS = np.arange(0, len(CHAINS), stride)
SUBSET_ROWS = np.flatnonzero(np.isin(CHAIN_OF_ROW, PICKED_CHAINS))

# 고른 사슬의 프레임만 시간 순으로 모은다. 이것이 추출 대상 전부다.
subset_files = [name for c in PICKED_CHAINS for name in CHAINS[c]]
subset_map = {name: i for i, name in enumerate(subset_files)}
sub_inputs = train_inputs.iloc[SUBSET_ROWS].reset_index(drop=True)
sub_index_matrix = np.asarray(
    [[subset_map[n] for n in row]
     for row in sub_inputs[IMAGE_COLUMNS].itertuples(index=False, name=None)], np.int64)
sub_wind = WIND_ALL[SUBSET_ROWS]
sub_wind_valid = WIND_VALID_ALL[SUBSET_ROWS]
sub_targets = TARGETS_ALL[SUBSET_ROWS]
# 폴드용 사슬 번호를 0..k-1 로 다시 매긴다
_remap = {c: i for i, c in enumerate(PICKED_CHAINS)}
SUB_CHAIN_ID = np.array([_remap[c] for c in CHAIN_OF_ROW[SUBSET_ROWS]])

total_frames = sum(len(c) for c in CHAINS)
print(f"train 전체   {len(train_inputs):,} 샘플 / 사슬 {len(CHAINS)}개 / 고유 이미지 {total_frames:,}")
print(f"서브셋       {len(SUBSET_ROWS):,} 샘플 ({len(SUBSET_ROWS)/len(train_inputs):.0%}) / "
      f"사슬 {len(PICKED_CHAINS)}개 (stride {stride}) / 고유 이미지 {len(subset_files):,}")
print(f"디코딩할 PNG {len(subset_files) * 2:,}장 "
      f"(전체판은 {(total_frames + 1408 + 4115) * 2:,}장)")
print(f"\n서브셋 사슬별 샘플 수: {np.bincount(SUB_CHAIN_ID).tolist()}")
print(f"타깃 평균 {sub_targets.mean():.1f} (전체 {TARGETS_ALL.mean():.1f}), "
      f"표준편차 {sub_targets.std():.1f} (전체 {TARGETS_ALL.std():.1f})")
print("  두 쌍이 비슷하면 서브셋이 전체를 대표한다는 뜻이다.")

## 2. Stonyhurst 투영 · 로그 표준화 · 동적 임계 (논문 §2.1–2.2)

전체판과 **완전히 같은 알고리즘**이고 출력 해상도만 36×72 로 줄였다. 논문 Fig. 4 의 (a)→(f):

off-limb 제거 → 경위도 역투영 → 지구 쪽 bounding box → 리샘플 → 동적 임계 이진맵.

표준화는 논문 Eq. 1 대로 **데이터셋 고정 상수**를 쓴다 (instance-wise 금지).
코로나홀 판정만 그날 quiet-Sun 대비 **동적 임계**다. 논문이 둘을 나눠 쓰는 그대로다.

면적은 `cos(lat)` 가중으로 **진짜 일면 면적**이다 (픽셀 격자는 경도 φ 에서 `cos φ` 만큼 단축된다).

> ⏱ 이 셀이 이 노트북에서 가장 오래 걸린다. 진행 로그와 함께 **전체판 추출 예상 시간**을
> 실측 속도로 환산해 출력한다 — 전체판을 돌릴지 판단하는 근거로 쓸 것.

In [ ]:
FINE_LAT, FINE_LON = FINE_GRID
FINE_CELLS = FINE_LAT * FINE_LON
assert MAP_LAT % FINE_LAT == 0 and MAP_LON % FINE_LON == 0
LEVEL_NAMES = [f"ch{c}" for c in CH_CUTS] + ["bright"]
N_LEVELS = len(LEVEL_NAMES)
BM_LEVEL = LEVEL_NAMES.index(f"ch{CH_CUTS[1]}")

_lat_edges = np.linspace(-LAT_RANGE_DEG, LAT_RANGE_DEG, MAP_LAT + 1)
_lon_edges = np.linspace(-LON_RANGE_DEG, LON_RANGE_DEG, MAP_LON + 1)
MAP_LAT_DEG = 0.5 * (_lat_edges[:-1] + _lat_edges[1:])
MAP_LON_DEG = 0.5 * (_lon_edges[:-1] + _lon_edges[1:])
FINE_LON_DEG = MAP_LON_DEG.reshape(FINE_LON, MAP_LON // FINE_LON).mean(axis=1)
DEG_PER_LON_BIN = float(2 * LON_RANGE_DEG / FINE_LON)

PIXEL_WEIGHT = np.repeat(np.cos(np.radians(MAP_LAT_DEG))[:, None], MAP_LON, axis=1)
PIXEL_WEIGHT = (PIXEL_WEIGHT / PIXEL_WEIGHT.sum()).astype(np.float32)
PIXEL_WEIGHT_FLAT = PIXEL_WEIGHT.ravel()
_fine_row = np.repeat(np.arange(FINE_LAT), MAP_LAT // FINE_LAT)
_fine_col = np.repeat(np.arange(FINE_LON), MAP_LON // FINE_LON)
FINE_CELL_ID = (_fine_row[:, None] * FINE_LON + _fine_col[None, :]).ravel().astype(np.int32)


def load_pair(name):
    planes = []
    for channel in CHANNELS:
        with Image.open(DATA_ROOT / "train" / channel / name) as image:
            planes.append(np.asarray(image.convert("L"), dtype=np.float32))
    return np.stack(planes)


def detect_disk(frame):
    """플레어에 둔감한 원반 검출. 배경과 원반 내부의 중간값을 임계로 쓴다."""
    plane = frame.mean(axis=0)
    background = np.percentile(plane, 2.0)
    interior = np.percentile(plane, 70.0)
    mask = plane > background + 0.35 * (interior - background)
    ys, xs = np.nonzero(mask)
    return float(ys.mean()), float(xs.mean()), float(math.sqrt(mask.sum() / math.pi))


_sampler_cache = {}


def stonyhurst_sampler(side, center_y, center_x, radius):
    """경위도 격자 -> 원본 픽셀의 이중선형 보간 인덱스/가중 (B0 회전 포함)."""
    key = (side, round(center_y), round(center_x), round(radius))
    if key in _sampler_cache:
        return _sampler_cache[key]
    _, cy, cx, r = key
    lat = np.radians(MAP_LAT_DEG)[:, None]
    lon = np.radians(MAP_LON_DEG)[None, :]
    b0 = math.radians(B0_DEG)
    x = np.cos(lat) * np.sin(lon)
    y = np.sin(lat) * math.cos(b0) - np.cos(lat) * np.cos(lon) * math.sin(b0)
    z = np.sin(lat) * math.sin(b0) + np.cos(lat) * np.cos(lon) * math.cos(b0)
    row, col = cy - r * y, cx + r * x
    inside = (z > 0) & (row >= 0) & (row <= side - 1) & (col >= 0) & (col <= side - 1)
    row = np.clip(row, 0, side - 1 - 1e-4); col = np.clip(col, 0, side - 1 - 1e-4)
    r0 = np.floor(row).astype(np.int64); dr = (row - r0).astype(np.float32)
    c0 = np.floor(col).astype(np.int64); dc = (col - c0).astype(np.float32)
    r1 = np.minimum(r0 + 1, side - 1); c1 = np.minimum(c0 + 1, side - 1)
    index = np.stack([(r0 * side + c0).ravel(), (r0 * side + c1).ravel(),
                      (r1 * side + c0).ravel(), (r1 * side + c1).ravel()])
    weight = np.stack([((1 - dr) * (1 - dc)).ravel(), ((1 - dr) * dc).ravel(),
                       (dr * (1 - dc)).ravel(), (dr * dc).ravel()]).astype(np.float32)
    weight *= inside.ravel().astype(np.float32)
    _sampler_cache[key] = (index, weight)
    while len(_sampler_cache) > 128:
        _sampler_cache.pop(next(iter(_sampler_cache)))
    return _sampler_cache[key]


def project(frame, center_y, center_x, radius):
    index, weight = stonyhurst_sampler(frame.shape[-1], center_y, center_x, radius)
    picked = frame.reshape(len(frame), -1)[:, index]
    return (picked * weight[None]).sum(axis=1).reshape(len(frame), MAP_LAT, MAP_LON)


CACHE_TAG = (f"{PROJ_CODE_VERSION}_{MAP_LAT}x{MAP_LON}_f{FINE_LAT}x{FINE_LON}"
             f"_n{len(subset_files)}_s{SUBSET_SAMPLES}")
MAP_PATH = CACHE_ROOT / f"qmap_{CACHE_TAG}.npy"
AREA_PATH = CACHE_ROOT / f"qarea_{CACHE_TAG}.npy"
CONST_PATH = CACHE_ROOT / f"qconst_{CACHE_TAG}.npy"

stage("extract_begin")
if MAP_PATH.exists() and AREA_PATH.exists() and CONST_PATH.exists():
    sub_maps = np.load(MAP_PATH, mmap_mode="r")
    sub_area = np.load(AREA_PATH)
    LOG_MEAN, LOG_STD = np.load(CONST_PATH)
    print(f"캐시 재사용: {MAP_PATH.name}")
    ELAPSED_PER_FRAME = None
else:
    # 논문 Eq. 1 의 고정 상수 — 서브셋에서 한 번만 잰다 (instance-wise 아님)
    probe = np.unique(np.linspace(0, len(subset_files) - 1, 300).astype(int))
    total, total_sq, count = np.zeros(2), np.zeros(2), 0
    for i in probe:
        frame = load_pair(subset_files[i])
        cy, cx, radius = detect_disk(frame)
        projected = project(frame, cy, cx, radius * DISK_MARGIN)
        valid = projected.min(axis=0) > 0
        values = np.log(projected[:, valid] + 1.0)
        total += values.sum(axis=1); total_sq += (values ** 2).sum(axis=1); count += int(valid.sum())
    LOG_MEAN = (total / count).astype(np.float32)
    LOG_STD = np.sqrt(np.maximum(total_sq / count - (total / count) ** 2, 1e-6)).astype(np.float32)
    np.save(CONST_PATH, np.stack([LOG_MEAN, LOG_STD]))

    sub_maps = np.lib.format.open_memmap(
        MAP_PATH, mode="w+", dtype=np.float16,
        shape=(len(subset_files), len(MAP_CHANNELS), MAP_LAT, MAP_LON))
    sub_area = np.zeros((len(subset_files), N_LEVELS, FINE_CELLS), np.float32)
    plane_of = {"193": 0, "211": 1}
    started = time.perf_counter()
    for index, name in enumerate(subset_files):
        frame = load_pair(name)
        cy, cx, radius = detect_disk(frame)
        projected = project(frame, cy, cx, radius * DISK_MARGIN)

        median = np.median(projected.reshape(2, -1), axis=1)[:, None, None]
        normalized = projected / np.maximum(median, 1e-3)
        binary = None
        for level, cut in enumerate(CH_CUTS):
            mask = ((normalized[0] <= cut) & (normalized[1] <= cut) if BM_CHANNEL == "and"
                    else normalized[plane_of[BM_CHANNEL]] <= cut)
            sub_area[index, level] = np.bincount(
                FINE_CELL_ID, weights=PIXEL_WEIGHT_FLAT * mask.ravel(), minlength=FINE_CELLS)
            if level == BM_LEVEL:
                binary = mask
        bright = (normalized[0] >= BRIGHT_CUT) & (normalized[1] >= BRIGHT_CUT)
        sub_area[index, -1] = np.bincount(
            FINE_CELL_ID, weights=PIXEL_WEIGHT_FLAT * bright.ravel(), minlength=FINE_CELLS)

        standardized = (np.log(np.maximum(projected, 0.0) + 1.0)
                        - LOG_MEAN[:, None, None]) / LOG_STD[:, None, None]
        for c, channel in enumerate(MAP_CHANNELS):
            sub_maps[index, c] = (binary if channel == "bm"
                                  else standardized[plane_of[channel]]).astype(np.float16)
        if (index + 1) % 500 == 0 or index + 1 == len(subset_files):
            done = time.perf_counter() - started
            print(f"  {index + 1}/{len(subset_files)}  {done:5.0f}s  "
                  f"({(index + 1) / done:.1f} 프레임/s)", flush=True)
    ELAPSED_PER_FRAME = (time.perf_counter() - started) / len(subset_files)
    sub_maps.flush(); del sub_maps
    sub_maps = np.load(MAP_PATH, mmap_mode="r")
    np.save(AREA_PATH, sub_area)

stage("extract_end")
print(f"\n추출 {STAGE_TIME['extract_end'] - STAGE_TIME['extract_begin']:.0f}s | "
      f"맵 {sub_maps.shape} {sub_maps.nbytes / 1024 ** 2:.0f}MB")
print(f"논문 Eq. 1 고정 상수: " +
      ", ".join(f"AIA {ch} mu={LOG_MEAN[i]:.3f} sigma={LOG_STD[i]:.3f}"
                for i, ch in enumerate(CHANNELS)))
print("\n일면 면적 비율 (서브셋 평균):")
for level, label in enumerate(LEVEL_NAMES):
    fraction = sub_area[:, level].sum(axis=1)
    print(f"  {label:10s} {fraction.mean():.4f} +- {fraction.std():.4f}"
          f"{'   ← 상수 0 이면 임계가 죽은 것' if fraction.std() < 1e-8 else ''}")
if ELAPSED_PER_FRAME:
    full = (total_frames + 1408 + 4115) * ELAPSED_PER_FRAME
    print(f"\n⏱ 실측 {1 / ELAPSED_PER_FRAME:.1f} 프레임/s → "
          f"전체판(15,662 프레임, 60x120) 추출 예상 **{full / 60:.0f}분** "
          f"(해상도 상승분 제외, 디코딩이 지배적이라 큰 차이는 없다)")

## 3. 격자 집계 · 탄도 정렬 · 중앙자오선 밴드 · Dataset

전체판과 동일하다. 요약만 다시 적으면:

- horizon $h$, 전달시간 $\tau=1\,\mathrm{AU}/v$ 의 소스 프레임 인덱스는 $t_{\rm raw}=19+(h-\tau)/6$
- $[0,19]$ 를 벗어난 분량은 **자전으로 경도에 넘긴다**:
  $t=\mathrm{clip}(t_{\rm raw},0,19)$, $\varphi=-\Omega\cdot 6\cdot(t_{\rm raw}-t)$
- 그 $(t,\varphi)$ 에서 **중앙자오선 ±10° 밴드**(논문 §4.3)의 위도별 면적을 뽑는다

### ⚠️ A0 vs A1 이 실제로 재는 것 — 과대해석 금지

`A0` 는 경도 외삽을 끈 것, `A1` 은 켠 것이다. 다만 **효과가 작게 나오는 것이 정상**이다.

기본값 `REFERENCE_TRANSIT_HOURS=108`, `BALLISTIC_OFFSETS=(-4,-2,0,2)` 에서
$t_{\rm raw}$ 의 범위는 $[-2,\,15]$ 다. 즉 **clip 이 걸리는 곳은 6h·12h horizon 의 음수 offset 뿐**이고,
나머지는 $\varphi=0$ 이라 두 팔이 같다. 아래 셀이 찍는 `lon` 표에서 0 이 아닌 칸이 몇 개인지 보면
이 비교가 얼마나 좁은지 바로 보인다.

또한 CH-GRU 의 gather 는 **시간 인덱스만** 쓰므로 경도 외삽의 영향을 받지 않는다.

**그래서 A1−A0 는 투영이 준 이득의 일부일 뿐이다.** 투영의 나머지 이득
(진짜 일면 면적 = `cos φ` 단축 제거, 중앙자오선 ±10° 밴드 정의)은 **A0 에도 이미 들어가 있어
이 사다리로는 분리되지 않는다.** 그걸 재려면 P7 의 픽셀 격자와 비교해야 하는데,
그건 추출 패스를 한 번 더 도는 일이라 이 축소판의 범위 밖이다.

$\tau$ 가 짧은 빠른 속도(600 km/s)에서 $t_{\rm raw}$ 가 19 를 넘는 것을 보려면
`BALLISTIC_SOURCE="speeds"` 로 바꿔야 한다 — 그때 A0/A1 격차가 커진다.

In [ ]:
def transit_hours(speed):
    return AU_KM / speed / 3600.0


def ballistic_plan(speeds=None, offsets=None):
    """(시간 인덱스, 경도 중심[deg]). 잘린 시간만큼을 자전으로 경도에 넘긴다."""
    if offsets is None:
        raw = np.stack([(HORIZONS - transit_hours(v)) / STEP_HOURS for v in speeds], axis=1)
    else:
        raw = ((HORIZONS - REFERENCE_TRANSIT_HOURS) / STEP_HOURS)[:, None] \
            + np.asarray(offsets, np.float64)[None, :]
    raw = 19.0 + raw
    index = np.clip(raw, 0.0, 19.0)
    longitude = -SIDEREAL_DEG_PER_HOUR * STEP_HOURS * (raw - index)
    if not USE_ROTATION_EXTRAPOLATION:
        longitude = np.zeros_like(longitude)
    return index.astype(np.float32), longitude.astype(np.float32)


def meridian_weight(longitude_center):
    low = longitude_center[..., None] - MERIDIAN_HALF_WIDTH_DEG
    high = longitude_center[..., None] + MERIDIAN_HALF_WIDTH_DEG
    edge_low = FINE_LON_DEG[None, None, :] - DEG_PER_LON_BIN / 2
    edge_high = FINE_LON_DEG[None, None, :] + DEG_PER_LON_BIN / 2
    overlap = np.clip(np.minimum(high, edge_high) - np.maximum(low, edge_low), 0.0, None)
    return (overlap / DEG_PER_LON_BIN).astype(np.float32)


def time_weight(index):
    lower = np.floor(index).astype(np.int64)
    upper = np.minimum(lower + 1, 19)
    fraction = (index - lower).astype(np.float32)
    weight = np.zeros(index.shape + (20,), np.float32)
    grid = np.indices(index.shape)
    np.add.at(weight, (grid[0], grid[1], lower), 1.0 - fraction)
    np.add.at(weight, (grid[0], grid[1], upper), fraction)
    return weight


def aggregate_lat(area, grid_lat, fold):
    block = area.reshape(len(area), -1, FINE_LAT, FINE_LON)
    block = block.reshape(len(area), -1, grid_lat, FINE_LAT // grid_lat, FINE_LON).sum(axis=3)
    if fold:
        block = block + block[:, :, ::-1, :]
        block = block[:, :, : (grid_lat + 1) // 2, :]
    return np.ascontiguousarray(block, dtype=np.float32)


def aggregate_full(area, grid_lat, grid_lon, fold):
    block = aggregate_lat(area, grid_lat, fold)
    block = block.reshape(len(area), block.shape[1], block.shape[2],
                          grid_lon, FINE_LON // grid_lon).sum(axis=4)
    return np.ascontiguousarray(block.reshape(len(area), block.shape[1], -1), np.float32)


def ballistic_features(lat_grid, index_matrix, chunk=1024):
    """시간 보간 가중 x 경도 밴드 가중을 한 커널로 합쳐 행렬곱 한 번으로 끝낸다."""
    kernel = (TIME_WEIGHT[:, :, :, None] * MERIDIAN_WEIGHT[:, :, None, :])
    kernel = kernel.reshape(-1, 20 * FINE_LON).T.astype(np.float32)
    n_levels, used_lat = lat_grid.shape[1], lat_grid.shape[2]
    output = np.empty((len(index_matrix), 12, N_SAMPLE_POINTS * n_levels * used_lat), np.float32)
    for start in range(0, len(index_matrix), chunk):
        rows = index_matrix[start:start + chunk]
        block = np.ascontiguousarray(lat_grid[rows].transpose(0, 2, 3, 1, 4))
        block = block.reshape(len(rows), n_levels * used_lat, 20 * FINE_LON)
        picked = (block @ kernel).reshape(len(rows), n_levels, used_lat, 12, N_SAMPLE_POINTS)
        output[start:start + chunk] = picked.transpose(0, 3, 4, 1, 2).reshape(len(rows), 12, -1)
    return output


def flatten_ch(grid, index_matrix):
    return grid[index_matrix].reshape(len(index_matrix), 20, -1).astype(np.float32)


def configure(**overrides):
    """설정을 바꾸고 파생 피처를 다시 만든다. 사다리는 이 함수로만 한다."""
    globals().update(overrides)
    global GRID_LAT, GRID_LON, USED_LAT, N_CELLS, LEVEL_INDEX, N_USED_LEVELS, CH_SEQ_DIM
    global TIME_INDEX, LON_CENTER, TIME_WEIGHT, MERIDIAN_WEIGHT, N_SAMPLE_POINTS
    global BALLISTIC_INDEX, N_SPEEDS, BALLISTIC_DIM, sub_grid, sub_lat

    GRID_LAT, GRID_LON = CH_GRID
    assert FINE_LAT % GRID_LAT == 0 and FINE_LON % GRID_LON == 0
    USED_LAT = (GRID_LAT + 1) // 2 if FOLD_LATITUDE else GRID_LAT
    N_CELLS = USED_LAT * GRID_LON
    LEVEL_INDEX = [LEVEL_NAMES.index(name) for name in USE_LEVELS]
    N_USED_LEVELS = len(LEVEL_INDEX)
    CH_SEQ_DIM = N_USED_LEVELS * N_CELLS
    sub_grid = aggregate_full(sub_area[:, LEVEL_INDEX], GRID_LAT, GRID_LON, FOLD_LATITUDE)
    sub_lat = aggregate_lat(sub_area[:, LEVEL_INDEX], GRID_LAT, FOLD_LATITUDE)

    if BALLISTIC_SOURCE == "window":
        TIME_INDEX, LON_CENTER = ballistic_plan(offsets=BALLISTIC_OFFSETS)
    else:
        TIME_INDEX, LON_CENTER = ballistic_plan(speeds=TRANSIT_SPEEDS)
    N_SAMPLE_POINTS = TIME_INDEX.shape[1]
    TIME_WEIGHT = time_weight(TIME_INDEX)
    MERIDIAN_WEIGHT = meridian_weight(LON_CENTER)
    BALLISTIC_DIM = N_SAMPLE_POINTS * N_USED_LEVELS * USED_LAT
    BALLISTIC_INDEX, _ = ballistic_plan(speeds=TRANSIT_SPEEDS)
    N_SPEEDS = len(TRANSIT_SPEEDS)


configure()

# ── Dataset ────────────────────────────────────────────────────────────────
STAT_NAMES = ["last", "mean4", "mean", "std", "min", "max", "slope", "last_minus_mean4", "range"]
NUM_STATS = len(STAT_NAMES)
_TIME_CENTERED = np.arange(20, dtype=np.float32) - 9.5
_TIME_DENOM = float((_TIME_CENTERED ** 2).sum())
USES_MAP = {"euv": True, "bm": True, "hybrid": True, "feature": False}
USES_WIND = {"euv": False, "bm": False, "hybrid": True, "feature": True}
VARIANT_MAP_CHANNELS = {
    "euv": [i for i, c in enumerate(MAP_CHANNELS) if c != "bm"],
    "bm": [i for i, c in enumerate(MAP_CHANNELS) if c == "bm"],
    "hybrid": list(range(len(MAP_CHANNELS))),
    "feature": [],
}


def shift_map(block, dy, dx):
    """가장자리 복제 평행이동. np.roll 과 달리 감아 넘기지 않는다 (자전 물리 보존)."""
    if dy == 0 and dx == 0:
        return block
    padded = np.pad(block, ((0, 0), (0, 0), (abs(dy), abs(dy)), (abs(dx), abs(dx))), mode="edge")
    y0, x0 = abs(dy) - dy, abs(dx) - dx
    return padded[:, :, y0:y0 + block.shape[2], x0:x0 + block.shape[3]]


def build_wind_stats(wind):
    last = wind[:, -1]
    mean4 = wind[:, -4:].mean(axis=1)
    slope = (wind - wind.mean(axis=1, keepdims=True)) @ _TIME_CENTERED / _TIME_DENOM
    return np.stack([last, mean4, wind.mean(axis=1), wind.std(axis=1), wind.min(axis=1),
                     wind.max(axis=1), slope, last - mean4,
                     wind.max(axis=1) - wind.min(axis=1)], axis=1).astype(np.float32)


def fit_stats(rows):
    """주어진 학습 행에서만 정규화/복원 통계를 산출한다."""
    wind, targets = sub_wind[rows], sub_targets[rows]
    ch_seq = flatten_ch(sub_grid, sub_index_matrix[rows]).reshape(-1, CH_SEQ_DIM)
    ballistic = ballistic_features(sub_lat, sub_index_matrix[rows]).reshape(-1, BALLISTIC_DIM)
    statistics = build_wind_stats(wind)
    residual = targets - wind[:, -1:]
    return {
        "wind_mean": float(wind.mean()), "wind_std": float(wind.std() + 1e-6),
        "diff_std": float(np.diff(wind, axis=1, prepend=wind[:, :1]).std() + 1e-6),
        "stats_mean": statistics.mean(axis=0), "stats_std": statistics.std(axis=0) + 1e-6,
        "ch_mean": ch_seq.mean(axis=0), "ch_std": ch_seq.std(axis=0) + 1e-8,
        "ballistic_mean": ballistic.mean(axis=0), "ballistic_std": ballistic.std(axis=0) + 1e-8,
        "residual_mean": residual.mean(axis=0), "residual_std": residual.std(axis=0) + 1e-6,
        "target_mean": targets.mean(axis=0), "target_std": targets.std(axis=0) + 1e-6,
        "clip_low": float(targets.min() * 0.95), "clip_high": float(targets.max() * 1.05),
    }


class SolarWindDataset(Dataset):
    def __init__(self, rows, stats, training=False):
        self.training, self.stats, self.rows = training, stats, rows
        self.index_matrix = sub_index_matrix[rows]
        self.maps = sub_maps if USES_MAP[MODEL_VARIANT] else None
        self.map_channels = VARIANT_MAP_CHANNELS[MODEL_VARIANT]
        self.gain_channels = [i for i, c in enumerate(self.map_channels)
                              if MAP_CHANNELS[c] != "bm"]
        wind, valid = sub_wind[rows], sub_wind_valid[rows]
        self.last_wind = np.ascontiguousarray(wind[:, -1]).astype(np.float32)
        self.wind_seq = np.stack([
            (wind - stats["wind_mean"]) / stats["wind_std"],
            np.diff(wind, axis=1, prepend=wind[:, :1]) / stats["diff_std"],
            valid], axis=2).astype(np.float32)
        self.wind_stats = ((build_wind_stats(wind) - stats["stats_mean"])
                           / stats["stats_std"]).astype(np.float32)
        self.ch_seq = ((flatten_ch(sub_grid, self.index_matrix) - stats["ch_mean"])
                       / stats["ch_std"]).astype(np.float32)
        self.ballistic = ((ballistic_features(sub_lat, self.index_matrix)
                           - stats["ballistic_mean"]) / stats["ballistic_std"]).astype(np.float32)
        self.targets = sub_targets[rows]

    def __len__(self):
        return len(self.rows)

    def _load_maps(self, item):
        block = np.asarray(self.maps[self.index_matrix[item]][:, self.map_channels], np.float32)
        if not (self.training and AUGMENT):
            return block
        if AUG_MAP_GAIN > 0 and self.gain_channels:
            block[:, self.gain_channels] *= np.float32(1.0 + np.random.normal(0, AUG_MAP_GAIN))
        if AUG_MAP_SHIFT > 0:
            block = shift_map(block, np.random.randint(-AUG_MAP_SHIFT, AUG_MAP_SHIFT + 1),
                              np.random.randint(-AUG_MAP_SHIFT, AUG_MAP_SHIFT + 1))
        return block

    def __getitem__(self, item):
        ch_seq, ballistic = self.ch_seq[item], self.ballistic[item]
        if self.training and AUGMENT and AUG_CH_NOISE > 0:
            ch_seq = ch_seq * (1 + np.random.normal(0, AUG_CH_NOISE, ch_seq.shape)).astype(np.float32)
            ballistic = ballistic * (1 + np.random.normal(0, AUG_CH_NOISE, ballistic.shape)
                                     ).astype(np.float32)
        return {"wind_seq": torch.from_numpy(self.wind_seq[item]),
                "wind_stats": torch.from_numpy(self.wind_stats[item]),
                "ch_seq": torch.from_numpy(np.ascontiguousarray(ch_seq)),
                "ballistic": torch.from_numpy(np.ascontiguousarray(ballistic)),
                "last_wind": torch.tensor(self.last_wind[item]),
                "target": torch.from_numpy(self.targets[item]),
                "maps": (torch.from_numpy(np.ascontiguousarray(self._load_maps(item)))
                         if self.maps is not None else torch.zeros(1))}


BATCH_KEYS = ("wind_seq", "wind_stats", "ch_seq", "ballistic", "maps", "last_wind")


def make_loader(rows, stats, shuffle, training, seed=SEED):
    options = dict(dataset=SolarWindDataset(rows, stats, training), batch_size=BATCH_SIZE,
                   shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY,
                   generator=torch.Generator().manual_seed(seed))
    if NUM_WORKERS > 0:
        options.update(persistent_workers=True, prefetch_factor=2)
    return DataLoader(**options)


print(f"격자 {GRID_LAT}x{GRID_LON} -> 셀 {N_CELLS} | ch_seq {CH_SEQ_DIM} | "
      f"탄도 {BALLISTIC_DIM} | gather 속도 {N_SPEEDS}개")
print(f"중앙자오선 밴드 +-{MERIDIAN_HALF_WIDTH_DEG:.0f}° = 경도 bin "
      f"{MERIDIAN_WEIGHT[0, 0].sum():.1f}개분 (bin 폭 {DEG_PER_LON_BIN:.1f}°)")
_speed_t, _speed_lon = ballistic_plan(speeds=TRANSIT_SPEEDS)
print("\n소스 인덱스 t / 경도 lon (19 = 마지막 관측, lon<0 = 아직 동쪽 = 자전 외삽):")
_table = pd.DataFrame(index=[f"{h}h" for h in HORIZONS])
for s, v in enumerate(TRANSIT_SPEEDS):
    _table[f"v{int(v)} t"] = _speed_t[:, s].round(1)
    _table[f"v{int(v)} lon"] = _speed_lon[:, s].round(1)
print(_table.to_string())

## 4. SpeedNet · 지표

논문 §3 Fig. 7 의 위상 그대로 — **Conv 블록 4개(Conv→ReLU→MaxPool) → LSTM(100) → FFNN(200)**.
필터 수만 16/32/64/64 로 줄였다 (논문·전체판은 32/64/128/128).

CNN 은 20 프레임에 가중치를 공유해 적용하고, 그 20개 특징 벡터가 LSTM 시퀀스가 된다.
논문은 하루 1장이라 시퀀스 축이 다르지만 CNN→LSTM→FFNN 구조는 동일하다.

| variant | 입력 | 논문 대응 |
|---|---|---|
| `feature` | 맵 없음. 격자 피처 + wind | **대조군** (P7 계열) |
| `bm` | 코로나홀 이진맵 1채널 | **SpeedNet-BM** (논문 최고: r 0.68 / RMSE 71.4) |
| `euv` | 표준화 EUV 2채널 | **SpeedNet-EUV** (r 0.65 / RMSE 75.9) |
| `hybrid` | 3채널 + wind + 탄도 피처 | 제출 후보 |

`bm`·`euv` 는 논문대로 **SW 속도를 입력에서 뺀다.** 그래서 단기 horizon 에서 크게 불리하다
(6h 는 지속성이 지배한다). 전체 RMSE 만 보지 말고 아래 표의 **42–72h 열**을 같이 볼 것 —
논문이 겨냥한 3~4일 리드가 그쪽이다.

손실은 대회 지표(horizon별 RMSE 평균)에 맞췄다. Threat Score 는 논문 Eq. 11 이고 진단용이다
(채점 대상 아님 — RMSE 최적 예측은 조건부 평균이라 분포가 좁아지고 TS 는 나빠지는 게 정상).

In [ ]:
class ConvBlock(nn.Module):
    """논문 Fig. 7 의 Conv 블록: Conv -> ReLU -> MaxPool."""

    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv2d(in_channels, out_channels, SPEEDNET_KERNEL,
                              padding=SPEEDNET_KERNEL // 2)
        self.pool = nn.MaxPool2d(2)

    def forward(self, x):
        return self.pool(F.relu(self.conv(x), inplace=True))


class SpeedNetBackbone(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        channels = (in_channels,) + tuple(SPEEDNET_FILTERS)
        self.blocks = nn.Sequential(*[ConvBlock(channels[i], channels[i + 1])
                                      for i in range(len(SPEEDNET_FILTERS))])
        with torch.no_grad():
            probe = self.blocks(torch.zeros(1, in_channels, MAP_LAT, MAP_LON))
        self.pool = (nn.AdaptiveAvgPool2d(SPEEDNET_POOL) if SPEEDNET_POOL is not None
                     else nn.Identity())
        shape = SPEEDNET_POOL if SPEEDNET_POOL is not None else probe.shape[-2:]
        self.output_dim = SPEEDNET_FILTERS[-1] * shape[0] * shape[1]
        self.conv_shape = tuple(probe.shape[-2:])

    def forward(self, maps):
        batch, frames = maps.shape[:2]
        return self.pool(self.blocks(maps.flatten(0, 1))).flatten(1).view(batch, frames, -1)


class SpeedNet(nn.Module):
    """논문 SpeedNet 을 대회 출력(12 horizon)에 맞춘 것. 절대 속도 (B, 12) 를 반환한다."""

    def __init__(self, stats):
        super().__init__()
        self.uses_map = USES_MAP[MODEL_VARIANT]
        self.uses_wind = USES_WIND[MODEL_VARIANT]
        self.shared_head = MODEL_VARIANT in ("hybrid", "feature")
        self.bidirectional = CH_BIDIRECTIONAL
        self.residual = RESIDUAL_OUTPUT
        shared_dim = 0

        if self.uses_map:
            self.backbone = SpeedNetBackbone(len(VARIANT_MAP_CHANNELS[MODEL_VARIANT]))
            self.lstm = nn.LSTM(self.backbone.output_dim, LSTM_UNITS, batch_first=True)
            self.map_dropout = nn.Dropout(DROPOUT)
            shared_dim += LSTM_UNITS
        if self.uses_wind:
            self.wind_gru = nn.GRU(3, 96, num_layers=2, batch_first=True)
            self.stats_encoder = nn.Sequential(
                nn.Linear(NUM_STATS, 128), nn.SELU(inplace=True),
                nn.Linear(128, 64), nn.SELU(inplace=True))
            shared_dim += 96 + 64

        head_extra = 0
        if self.shared_head:
            directions = 2 if self.bidirectional else 1
            self.ch_gru = nn.GRU(CH_SEQ_DIM, CH_HIDDEN, num_layers=2, batch_first=True,
                                 bidirectional=self.bidirectional)
            self.ch_dropout = nn.Dropout(DROPOUT)
            shared_dim += CH_HIDDEN * directions
            self.gather_project = nn.Sequential(
                nn.Linear(CH_HIDDEN * directions, GATHER_DIM), nn.ReLU(inplace=True))
            self.horizon_embedding = nn.Parameter(torch.randn(12, HORIZON_EMBED) * 0.1)
            head_extra = GATHER_DIM * N_SPEEDS + BALLISTIC_DIM + HORIZON_EMBED

        self.ffnn = nn.Sequential(
            nn.Linear(shared_dim + head_extra, FFNN_UNITS), nn.ReLU(inplace=True),
            nn.Dropout(DROPOUT),
            nn.Linear(FFNN_UNITS, FFNN_UNITS // 2), nn.ReLU(inplace=True), nn.Dropout(DROPOUT),
            nn.Linear(FFNN_UNITS // 2, 1 if self.shared_head else 12))

        lower = np.floor(BALLISTIC_INDEX).astype(np.int64)
        self.register_buffer("gather_lower", torch.as_tensor(lower))
        self.register_buffer("gather_upper", torch.as_tensor(np.minimum(lower + 1, 19)))
        self.register_buffer("gather_weight",
                             torch.as_tensor(BALLISTIC_INDEX - lower, dtype=torch.float32))
        self.register_buffer("out_center", torch.as_tensor(
            stats["residual_mean"] if RESIDUAL_OUTPUT else stats["target_mean"]))
        self.register_buffer("out_scale", torch.as_tensor(
            stats["residual_std"] if RESIDUAL_OUTPUT else stats["target_std"]))
        if VERBOSE_MODEL:
            print(f"[{MODEL_VARIANT}] "
                  f"{'conv ' + str(self.backbone.conv_shape) if self.uses_map else '맵 없음'} "
                  f"shared={shared_dim} head_extra={head_extra}")

    def forward(self, wind_seq, wind_stats, ch_seq, ballistic, maps, last_wind):
        parts = []
        if self.uses_map:
            _, (hidden, _) = self.lstm(self.backbone(maps))
            parts.append(self.map_dropout(hidden[-1]))
        if self.uses_wind:
            _, wind_hidden = self.wind_gru(wind_seq)
            parts += [F.relu(wind_hidden[-1]), self.stats_encoder(wind_stats)]

        if not self.shared_head:
            z = self.ffnn(torch.cat(parts, dim=1))          # 논문 원형: FFNN -> Dense(12)
        else:
            ch_sequence, ch_hidden = self.ch_gru(ch_seq)
            ch_last = (torch.cat([ch_hidden[-2], ch_hidden[-1]], dim=1)
                       if self.bidirectional else ch_hidden[-1])
            parts.append(self.ch_dropout(F.relu(ch_last)))
            shared = torch.cat(parts, dim=1)
            batch = shared.shape[0]
            projected = self.gather_project(ch_sequence)
            weight = self.gather_weight.unsqueeze(-1)
            gathered = (projected[:, self.gather_lower] * (1.0 - weight)
                        + projected[:, self.gather_upper] * weight)
            z = self.ffnn(torch.cat([
                shared.unsqueeze(1).expand(batch, 12, shared.shape[1]),
                self.horizon_embedding.unsqueeze(0).expand(batch, 12, HORIZON_EMBED),
                gathered.flatten(2), ballistic], dim=2)).squeeze(-1)

        prediction = z * self.out_scale + self.out_center
        return prediction + last_wind.unsqueeze(1) if self.residual else prediction


# ── 지표 ───────────────────────────────────────────────────────────────────
def official_rmse(y_true, y_pred):
    """대회 지표: horizon 별 RMSE 를 낸 뒤 평균 (pooled RMSE 와 다르다)."""
    per_horizon = np.sqrt(np.mean((y_pred - y_true) ** 2, axis=0))
    return float(per_horizon.mean()), per_horizon


def loss_function(prediction, target):
    error = (prediction - target) / LOSS_SCALE
    if LOSS_KIND == "mse":
        return (error ** 2).mean()                          # 논문 Table 4
    return torch.sqrt((error ** 2).mean(dim=0) + LOSS_EPSILON).mean()


def threat_score(y_true, y_pred, last_wind):
    """논문 Eq. 11. HSE = 24h(4스텝) 전 대비 50 km/s 이상 증가."""
    def labels(series):
        extended = np.concatenate(
            [np.repeat(last_wind[:, None], HSE_LOOKBACK, axis=1), series], axis=1)
        return (series - extended[:, :series.shape[1]]) >= HSE_THRESHOLD
    observed, predicted = labels(y_true), labels(y_pred)
    denominator = int(np.sum(observed | predicted))
    return (int(np.sum(observed & predicted)) / denominator) if denominator else float("nan")


def correlation(y_true, y_pred):
    """논문 Eq. 10. horizon 별로 내고 평균."""
    a = y_pred - y_pred.mean(axis=0, keepdims=True)
    b = y_true - y_true.mean(axis=0, keepdims=True)
    return float(((a * b).sum(axis=0)
                  / (np.sqrt((a ** 2).sum(axis=0) * (b ** 2).sum(axis=0)) + 1e-12)).mean())


@torch.no_grad()
def predict_with(model, loader, clip_low, clip_high):
    model.eval()
    out = []
    for batch in loader:
        moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY) for k in BATCH_KEYS}
        with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
            prediction = model(moved["wind_seq"], moved["wind_stats"], moved["ch_seq"],
                               moved["ballistic"], moved["maps"], moved["last_wind"])
        out.append(prediction.float().clamp(clip_low, clip_high).cpu().numpy())
    return np.concatenate(out).astype(np.float64)


VERBOSE_MODEL = True
_probe = SpeedNet(fit_stats(np.arange(len(SUBSET_ROWS)))).to(DEVICE)
print("feature 변형 파라미터:", f"{sum(p.numel() for p in _probe.parameters()):,}")
del _probe; gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()
VERBOSE_MODEL = False

## 5. 사다리 — 논문 두 변형 vs 대조군

논문 §2.3 (Fig. 5b) 처럼 **연속 구간(사슬)** 단위로 폴드를 나눈다. 무작위 분할은 쓰지 않는다 —
인접 시각의 태양 맵이 거의 같아서 학습·검증이 사실상 같은 자료가 되기 때문이다.

**모든 변형이 같은 폴드·같은 시드를 쓴다.** 페어드 비교라야 표본이 작아도 순서를 읽을 수 있다.

epoch 선택은 전 폴드 평균 곡선에서 **한 번만** 한다(`fixed`). 논문의 meta-learning 체크포인트
(폴드마다 최저점)도 함께 찍는데(`meta`), 그건 검증셋으로 고른 값이라 항상 낙관적이다.
두 값의 차이가 곧 검증 과적합의 크기다.

마지막 출력이 **판정**과 **전체판 예상 시간**이다.

In [ ]:
def build_folds(n_folds):
    """사슬을 샘플 수가 고르게 되도록 폴드에 배정한다 (논문 Fig. 5b 의 연속 구간)."""
    counts = np.bincount(SUB_CHAIN_ID, minlength=len(PICKED_CHAINS))
    assignment, loads = np.zeros(len(PICKED_CHAINS), np.int64), np.zeros(n_folds, np.int64)
    for chain in np.argsort(counts)[::-1]:
        fold = int(np.argmin(loads))
        assignment[chain] = fold
        loads[fold] += counts[chain]
    return [(np.flatnonzero(assignment[SUB_CHAIN_ID] != f),
             np.flatnonzero(assignment[SUB_CHAIN_ID] == f)) for f in range(n_folds)]


FOLDS = build_folds(LADDER_FOLDS)
print("폴드별 (학습, 평가) 샘플 수:", [(len(a), len(b)) for a, b in FOLDS])

# 같은 폴드에서 잰 기준선. 데이터 크기가 달라도 "persistence 대비 개선폭"은 비교적 잘 전이된다.
BASELINE = {}
for name in ("persistence", "climatology"):
    scores = []
    for train_rows, evaluate_rows in FOLDS:
        truth = sub_targets[evaluate_rows]
        guess = (np.repeat(sub_wind[evaluate_rows, -1:], 12, axis=1) if name == "persistence"
                 else np.repeat(sub_targets[train_rows].mean(axis=0)[None], len(truth), axis=0))
        scores.append(official_rmse(truth, guess)[0])
    BASELINE[name] = float(np.mean(scores))
    print(f"{name:12s} CV RMSE {BASELINE[name]:7.3f}")


def train_run(train_rows, evaluate_rows, epochs, seed=SEED):
    """고정 epoch 학습. 매 epoch 의 홀드아웃 horizon별 RMSE 를 기록만 한다."""
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    stats = fit_stats(train_rows)
    train_loader = make_loader(train_rows, stats, True, True, seed)
    evaluate_loader = make_loader(evaluate_rows, stats, False, False, seed)
    truth = sub_targets[evaluate_rows]

    model = SpeedNet(stats).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE,
                                  weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(DEVICE.type, enabled=USE_AMP)

    curve = np.zeros((epochs, 12))
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            moved = {k: batch[k].to(DEVICE, non_blocking=PIN_MEMORY)
                     for k in BATCH_KEYS + ("target",)}
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE.type, enabled=USE_AMP):
                prediction = model(moved["wind_seq"], moved["wind_stats"], moved["ch_seq"],
                                   moved["ballistic"], moved["maps"], moved["last_wind"])
            loss = loss_function(prediction.float(), moved["target"])
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer); scaler.update()
        scheduler.step()
        curve[epoch] = official_rmse(
            truth, predict_with(model, evaluate_loader,
                                stats["clip_low"], stats["clip_high"]))[1]
    final = predict_with(model, evaluate_loader, stats["clip_low"], stats["clip_high"])
    extra = (correlation(truth, final), threat_score(truth, final, sub_wind[evaluate_rows, -1]))
    del model, train_loader, evaluate_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return curve, extra


# 라벨은 ASCII 로 둔다 — 플랫폼에 한글 폰트가 없어 matplotlib 경고가 출력을 뒤덮고
# pandas to_string 의 열 정렬도 CJK 폭 때문에 깨진다 (HANDOFF §7.5).
LADDER = [
    ("A0 feature no-extrap", dict(MODEL_VARIANT="feature", FFNN_UNITS=200,
                                  USE_ROTATION_EXTRAPOLATION=False)),
    ("A1 feature control",   dict(MODEL_VARIANT="feature", FFNN_UNITS=200,
                                  USE_ROTATION_EXTRAPOLATION=True)),
    ("B  SpeedNet-BM",       dict(MODEL_VARIANT="bm", FFNN_UNITS=400)),
    ("C  SpeedNet-EUV",      dict(MODEL_VARIANT="euv", FFNN_UNITS=200)),
    ("D  hybrid",            dict(MODEL_VARIANT="hybrid", FFNN_UNITS=200)),
]

stage("ladder_begin")
RESULTS = []
for label, overrides in LADDER:
    configure(**overrides)
    started = time.perf_counter()
    curves, extras = [], []
    for f, (train_rows, evaluate_rows) in enumerate(FOLDS):
        curve, extra = train_run(train_rows, evaluate_rows, LADDER_EPOCHS)
        curves.append(curve); extras.append(extra)
        print(f"  {label:20s} fold {f + 1}/{len(FOLDS)}  "
              f"best {curve.mean(axis=1).min():7.3f}  "
              f"({time.perf_counter() - started:5.0f}s)", flush=True)
    curves = np.stack(curves)                                   # (fold, epoch, 12)
    mean_curve = curves.mean(axis=0)
    best_epoch = int(np.argmin(mean_curve.mean(axis=1)))
    per_horizon = mean_curve[best_epoch]
    fold_scores = curves[:, best_epoch].mean(axis=1)
    RESULTS.append({
        "label": label, "curves": curves, "mean": mean_curve, "best_epoch": best_epoch,
        "fixed": float(per_horizon.mean()),
        "meta": float(curves.mean(axis=2).min(axis=1).mean()),
        "per_horizon": per_horizon,
        "short": float(per_horizon[:6].mean()), "long": float(per_horizon[6:].mean()),
        "r": float(np.mean([e[0] for e in extras])),
        "ts": float(np.nanmean([e[1] for e in extras])),
        "se": float(fold_scores.std(ddof=1) / np.sqrt(len(fold_scores))),
        "seconds": time.perf_counter() - started,
        "is_cnn": overrides["MODEL_VARIANT"] != "feature",
    })
stage("ladder_end")

summary = pd.DataFrame([{
    "variant": r["label"], "epoch": r["best_epoch"] + 1,
    "CV(fixed)": r["fixed"], "CV(meta)": r["meta"],
    "6-36h": r["short"], "42-72h": r["long"], "72h": r["per_horizon"][-1],
    "r": r["r"], "TS": r["ts"], "fold_se": r["se"], "sec": r["seconds"]} for r in RESULTS])
print("\n" + "=" * 104)
print(summary.round(3).to_string(index=False))
print("=" * 104)
print(f"기준선: persistence {BASELINE['persistence']:.3f} / "
      f"climatology {BASELINE['climatology']:.3f}  (같은 폴드에서 잰 값)")
summary.to_csv(OUTPUT_DIR / "quick_ablation.csv", index=False)

# ── 판정 ───────────────────────────────────────────────────────────────────
feature_best = min((r for r in RESULTS if not r["is_cnn"]), key=lambda r: r["fixed"])
cnn_best = min((r for r in RESULTS if r["is_cnn"]), key=lambda r: r["fixed"])
delta = cnn_best["fixed"] - feature_best["fixed"]
noise = 2 * max(feature_best["se"], cnn_best["se"])
delta_long = cnn_best["long"] - feature_best["long"]

print("\n" + "#" * 104)
print(f"# 최고 CNN : {cnn_best['label']:20s} {cnn_best['fixed']:7.3f}  "
      f"(42-72h {cnn_best['long']:6.2f})")
print(f"# 대조군   : {feature_best['label']:20s} {feature_best['fixed']:7.3f}  "
      f"(42-72h {feature_best['long']:6.2f})")
print(f"# 차이     : {delta:+7.3f} km/s  (42-72h {delta_long:+6.2f})   "
      f"노이즈 바닥 +-{noise:.3f}")
print("#")
if delta < -noise:
    print("# 판정: CNN 이 대조군을 이겼다. 표본 1/3 · 필터 절반이라는 불리한 조건에서 이겼으므로")
    print("#       전체 데이터에서는 격차가 더 벌어질 가능성이 높다.")
    print("#       -> code_p8.ipynb (전체판) 를 돌릴 가치가 있다.")
elif delta < noise:
    print("# 판정: 구분 불가. 차이가 노이즈 바닥 안이다.")
    print("#       -> 시간이 넉넉하면 SUBSET_SAMPLES / LADDER_FOLDS 를 키워 다시 재거나 전체판으로.")
    print("#          시간이 없으면 P3(Public 58.8028) 을 유지할 것.")
else:
    print("# 판정: CNN 이 대조군에 졌다. 데이터 부족만으로 설명하기 어려운 격차다.")
    print("#       HANDOFF §4 의 P6 실패(val 은 이겼으나 Public 에서 1.39 패)와 같은 방향이다.")
    print("#       -> 전체판 CNN 에 시간을 쓰지 말 것. P3 유지 또는 feature 계열(P7)에 집중.")
print("#")
extrapolation = sum(r["seconds"] for r in RESULTS if r["is_cnn"]) / max(len(FOLDS), 1) \
    * (len(train_inputs) / max(len(SUBSET_ROWS), 1)) * (60 * 120) / (MAP_LAT * MAP_LON) \
    * (128 / 64) * (20 / max(LADDER_EPOCHS, 1)) * 2 / 60
print(f"# 전체판 사다리(CNN 3종 x 2fold x 20epoch, 60x120, 필터 2배) 예상 "
      f"약 {extrapolation:.0f}분 + 추출")
print(f"# 이 노트북 총 소요 {(STAGE_TIME['ladder_end'] - STAGE_TIME['start']) / 60:.1f}분 "
      f"(추출 {(STAGE_TIME['extract_end'] - STAGE_TIME['extract_begin']) / 60:.1f}분 포함)")
print("#" * 104)

figure, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for r in RESULTS:
    axes[0].plot(np.arange(1, LADDER_EPOCHS + 1), r["mean"].mean(axis=1), label=r["label"])
axes[0].axhline(BASELINE["persistence"], color="gray", ls="--", lw=1, label="persistence")
axes[0].set_xlabel("epoch"); axes[0].set_ylabel("CV RMSE [km/s]")
axes[0].legend(fontsize=7); axes[0].grid(alpha=0.3)
axes[0].set_title("learning curve (still falling = need more epochs)")
for r in RESULTS:
    axes[1].plot(HORIZONS, r["per_horizon"], "o-", label=r["label"])
axes[1].set_xlabel("horizon [hour]"); axes[1].set_ylabel("RMSE [km/s]")
axes[1].legend(fontsize=7); axes[1].grid(alpha=0.3); axes[1].set_title("RMSE by horizon")
plt.tight_layout(); plt.savefig(OUTPUT_DIR / "quick.png", dpi=140); plt.show()